# Notebook 2: Luyện Đan Fine-tune 3 Agents (Unsloth + LoRA Trên Colab)
---
**Hướng dẫn thao tác:**
1. Mở [Google Colab](https://colab.research.google.com/) -> Vào `Runtime` -> `Change runtime type` -> Chọn **T4 GPU**.
2. Upload 3 file train (`agent1_scope_train.jsonl`, `agent2_criteria_train.jsonl`, `agent3_pico_train.jsonl`) lên mục Files bên trái Colab.
3. Chạy lần lượt các Cell dưới đây.

In [ ]:
# Cell 1: Cài đặt Unsloth (Nhanh gấp đôi, tiết kiệm 70% VRAM)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# Cell 2: Tải Base Model Llama-3-8B 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Đã gắn LoRA thành công!")

In [ ]:
# Cell 3: Chuẩn bị hàm format Alpaca cho Llama 3
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

print("✅ Đã cấu hình Formatter!")

In [ ]:
# Cell 4: Hàm huấn luyện và lưu LoRA cho từng Agent
def train_agent_lora(train_file, adapter_name, max_steps=120):
    print(f"\n🚀 BẮT ĐẦU HUẤN LUYỆN: {adapter_name} ({train_file})")
    dataset = load_dataset("json", data_files=train_file, split="train")
    dataset = dataset.map(format_prompts, batched=True)
    
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            max_steps = max_steps,
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = f"outputs_{adapter_name}",
        ),
    )
    trainer.train()
    model.save_pretrained(adapter_name)
    tokenizer.save_pretrained(adapter_name)
    print(f"🎉 Đã huấn luyện xong và lưu LoRA: {adapter_name}")

# Bạn có thể chạy train từng con hoặc chạy cả 3:
train_agent_lora("agent1_scope_train.jsonl", "lora_agent1_scope", max_steps=120)
train_agent_lora("agent2_criteria_train.jsonl", "lora_agent2_criteria", max_steps=120)
train_agent_lora("agent3_pico_train.jsonl", "lora_agent3_pico", max_steps=120)

In [ ]:
# Cell 5: Test thử trực tiếp mô hình vừa train
FastLanguageModel.for_inference(model)

test_prompt = prompt_template.format(
    "Evaluate the research scope and suggest refinements.",
    "Domain: Robotics\nTopic: Dùng AI cho tay gắp robot\nIdea: Dùng Deep RL huấn luyện robot gắp mọi đồ vật trên đời",
    ""
)
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)

print("\n--- KẾT QUẢ AI VỪA HUẤN LUYỆN SINH RA ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])